In [2]:
"""
Disneyland One-Day Optimal Itinerary Planning with Meet-and-Greet
========================================
Entry Time: 08:30
Rules:
  - Each zone can only be entered once
  - After finishing all rides in a zone, immediately go to the nearest show
    (wait ≤ 15 minutes is considered "immediate")
  - Choose the show with the shortest wait and highest rating
  - Show duration ~ N(15, 3) minutes (minimum 5 minutes)
  - ONE meet-and-greet maximum (location must be current or next zone)
  - Meet-and-greet duration ~ N(15, 3) minutes (minimum 5 minutes)
  - Meet-and-greet total time = queue time + duration
  - Meet-and-greet must occur within operating hours
  - Total score = ride score + show score + best shop score + meet-and-greet score
  - After 240 minutes of play, schedule a 30-minute meal (lunch)
  - Force a 30-minute dinner between 17:30 and 19:30 (same logic as lunch)
  - Walking time + 3 min entry/exit time is automatically included between zones
"""

import pandas as pd
import numpy as np

# ─── Constants (modified: shop and checkin time ×1.5, shop score ×1.5) ───
START_MINUTE  = 8 * 60 + 33  # 08:30
LIMIT         = 647
MEAL_TRIGGER  = 200          # 12:30 (4 hours after entry)
MEAL_TIME     = 30
DINNER_TRIGGER = 500         # 17:30 (9 hours after entry)
DINNER_DUR    = 30
DINNER_END_LIMIT = 680       # 19:30

SHOP_TIME     = 23           # 15 * 1.5 rounded
CHECKIN_TIME  = 15           # 10 * 1.5

SHOW_DUR_MEAN = 15
SHOW_DUR_STD  = 3
MEETGREET_DUR_MEAN = 15
MEETGREET_DUR_STD  = 3
MAX_WAIT      = 15
WALK_SPEED_MIN_PER_KM = 15
ZONE_IN_OUT_EXTRA = 3

# Zone order corresponding to distance matrix (fixed order)
fixed_zone_order = [
    "米奇大街",
    "奇想花园",
    "明日世界",
    "梦幻世界",
    "宝藏湾",
    "皮克斯玩具总动员",
    "探险岛"
]
zone_list = fixed_zone_order.copy()

# 7×7 distance matrix (km) - same for all scenarios
dist_matrix_km = np.array([
    [0.00, 0.09, 0.35, 0.10, 0.44, 0.44, 0.54],
    [0.09, 0.00, 0.28, 0.17, 0.53, 0.52, 0.60],
    [0.35, 0.28, 0.00, 0.36, 0.76, 0.63, 0.64],
    [0.10, 0.17, 0.36, 0.00, 0.40, 0.35, 0.44],
    [0.44, 0.53, 0.76, 0.40, 0.00, 0.31, 0.47],
    [0.44, 0.52, 0.63, 0.35, 0.31, 0.00, 0.17],
    [0.54, 0.60, 0.64, 0.44, 0.47, 0.17, 0.00]
])

zone2idx = {z: i for i, z in enumerate(zone_list)}

def get_walk_time(from_zone, to_zone):
    if from_zone not in zone2idx or to_zone not in zone2idx:
        return 0
    i = zone2idx[from_zone]
    j = zone2idx[to_zone]
    dist = dist_matrix_km[i, j]
    return int(round(dist * WALK_SPEED_MIN_PER_KM))

# ─── 1. Load Show Data (once) ───────────────────────────────────────────
shows_raw = pd.read_excel("演出.xlsx", header=0)
shows_raw.columns = ["园区", "演出", "评分", "场次"]
shows_raw["园区"] = shows_raw["园区"].ffill()

def parse_times(s):
    if pd.isna(s):
        return []
    s = str(s).replace("：", ":").replace("，", ",").replace("\u3001", ",")
    out = []
    for t in s.split(","):
        t = t.strip()
        if not t:
            continue
        pm = "下午" in t
        t2 = t.replace("上午", "").replace("下午", "").strip()
        try:
            h, m = map(int, t2.split(":"))
        except ValueError:
            continue
        if pm and h < 12:
            h += 12
        off = h * 60 + m - START_MINUTE
        if off >= 0:
            out.append(off)
    return sorted(out)

zone_shows = {}
for _, row in shows_raw.iterrows():
    z = row["园区"]
    sc = pd.to_numeric(row["评分"], errors="coerce")
    if pd.isna(sc):
        sc = 0.0
    zone_shows.setdefault(z, []).append(
        (row["演出"], float(sc), parse_times(row["场次"]))
    )

def sample_dur():
    return max(5, int(round(np.random.normal(SHOW_DUR_MEAN, SHOW_DUR_STD))))

def sample_meetgreet_dur():
    return max(5, int(round(np.random.normal(MEETGREET_DUR_MEAN, MEETGREET_DUR_STD))))

def pick_show(zone, finish_offset):
    if zone not in zone_shows:
        return None
    best = None
    best_key = (MAX_WAIT + 1, -99.0)
    for (name, sc, slots) in zone_shows[zone]:
        if not slots:
            continue
        idx = int(np.searchsorted(slots, finish_offset))
        if idx >= len(slots):
            continue
        slot = slots[idx]
        wait = slot - finish_offset
        if wait > MAX_WAIT:
            continue
        key = (wait, -sc)
        if key < best_key:
            best_key = key
            best = (name, sc, slot, wait, sample_dur())
    return best

# ─── 2. Load Meet-and-Greet Data (once) ───────────────────────────────────
def parse_meetgreet_time_range(s):
    if pd.isna(s):
        return None, None
    s = str(s).replace("：", ":").replace("；", ":").replace(" ", "")
    parts = s.split("-")
    if len(parts) != 2:
        return None, None
    try:
        start_h, start_m = map(int, parts[0].split(":"))
        end_h, end_m = map(int, parts[1].split(":"))
        start_off = start_h * 60 + start_m - START_MINUTE
        end_off = end_h * 60 + end_m - START_MINUTE
        return max(0, start_off), end_off
    except:
        return None, None

meetgreet_raw = pd.read_excel("见面会.xlsx", header=0)
meetgreet_raw.columns = ["见面会名称", "地点", "开放时间", "评分", "排队时间"]

meetgreets = []
for _, row in meetgreet_raw.iterrows():
    name = row["见面会名称"]
    zone = row["地点"]
    score = pd.to_numeric(row["评分"], errors="coerce")
    queue = pd.to_numeric(row["排队时间"], errors="coerce")
    
    if pd.isna(score) or pd.isna(queue):
        continue
    
    start_off, end_off = parse_meetgreet_time_range(row["开放时间"])
    if start_off is None or end_off is None:
        continue
    
    meetgreets.append({
        "名称": name,
        "地点": zone,
        "评分": float(score),
        "排队时间": int(queue),
        "开始时间": start_off,
        "结束时间": end_off
    })

print(f"Loaded {len(meetgreets)} meet-and-greets (common to all scenarios)")

# ─── 3. Load Shop Data (once) ──────────────────────────────────────────────
shops = pd.read_excel("商店.xlsx")
shops.columns = ["园区", "商店", "评分"]
shops["园区"] = shops["园区"].ffill()
shops = shops.dropna(subset=["商店", "评分"])
shops["评分"] = pd.to_numeric(shops["评分"], errors="coerce")
shops = shops.dropna(subset=["评分"])

best_shop = shops.sort_values("评分", ascending=False).groupby("园区").first()
# ★ Shop score multiplied by 1.5
shop_score_map = {z: float(best_shop.loc[z, "评分"]) * 1.5 for z in best_shop.index}
shop_name_map = {z: best_shop.loc[z, "商店"] for z in best_shop.index}

# ===========================================================================
#  Simulation function for a given project file
# ===========================================================================
def run_scenario(project_csv_path, scenario_name, reset_seed=True):
    """Run the complete optimization for one project file.
       Returns a dict with results and best itinerary data."""
    
    if reset_seed:
        np.random.seed(42)   # ensure same random draws across scenarios
    
    # Read projects
    projects = pd.read_csv(project_csv_path)
    projects.columns = ["园区", "项目", "评分", "时间"]
    projects["园区"] = projects["园区"].ffill()
    projects = projects.dropna(subset=["项目", "评分", "时间"])
    projects["评分"] = pd.to_numeric(projects["评分"], errors="coerce")
    projects["时间"] = pd.to_numeric(projects["时间"], errors="coerce")
    projects = projects.dropna(subset=["评分", "时间"])
    
    # Build item list in zone order
    items = []
    for z in fixed_zone_order:
        zp = projects[projects["园区"] == z].sort_values(["评分", "时间"], ascending=[False, True])
        for _, row in zp.iterrows():
            score = pd.to_numeric(row["评分"], errors="coerce")
            time = pd.to_numeric(row["时间"], errors="coerce")
            if pd.isna(score) or pd.isna(time):
                continue
            items.append({
                "园区": row["园区"], "名称": row["项目"],
                "评分": float(score), "时间": int(time)
            })
    
    n = len(items)
    times_arr = np.array([x["时间"] for x in items], dtype=np.int32)
    scores_arr = np.array([x["评分"] for x in items], dtype=np.float64)
    zones_arr = np.array([x["园区"] for x in items], dtype=object)
    
    suffix_score = np.zeros(n + 1)
    for i in range(n - 1, -1, -1):
        suffix_score[i] = suffix_score[i + 1] + scores_arr[i]
    
    # Global variables for best solution (reset for this scenario)
    best_score_g = -1.0
    best_choice_g = []
    best_shows_g = []
    best_meetgreet_g = None
    best_meetgreet_insert_g = None
    best_total_time_g = 0
    
    # DFS definition (uses outer nonlocal variables)
    def dfs(i, chosen, shows, visited, proj_time, proj_score, show_score,
            meetgreet_data, meetgreet_inserted_at, lunch_done, dinner_done):
        nonlocal best_score_g, best_choice_g, best_shows_g, best_meetgreet_g, best_meetgreet_insert_g, best_total_time_g
        
        # compute current shop & meet-and-greet scores
        # Shop score already multiplied by 1.5 in shop_score_map
        shop_sc = sum(shop_score_map.get(z, 0) for z in visited)
        mg_sc = meetgreet_data["评分"] if (meetgreet_data and meetgreet_inserted_at) else 0
        
        # optimistic pruning
        optimistic = proj_score + show_score + shop_sc + mg_sc + suffix_score[i]
        if optimistic <= best_score_g:
            return
        
        # mandatory times (SHOP_TIME and CHECKIN_TIME already ×1.5)
        shop_t = len(visited) * SHOP_TIME
        chk_t = len(visited) * CHECKIN_TIME
        mg_t = (meetgreet_data["排队时间"] + meetgreet_data["duration"]) if (meetgreet_data and meetgreet_inserted_at) else 0
        
        # Lunch decision
        lunch_t = 0
        new_lunch_done = lunch_done
        if not lunch_done and proj_time > MEAL_TRIGGER:
            lunch_t = MEAL_TIME
            new_lunch_done = True
        
        # Dinner decision
        dinner_t = 0
        new_dinner_done = dinner_done
        if not dinner_done and proj_time > DINNER_TRIGGER:
            if proj_time + DINNER_DUR <= DINNER_END_LIMIT:
                dinner_t = DINNER_DUR
                new_dinner_done = True
        
        total_t = proj_time + shop_t + chk_t + mg_t + lunch_t + dinner_t
        if total_t > LIMIT:
            return
        
        # leaf node
        if i == n:
            sc = proj_score + show_score + shop_sc + mg_sc
            if sc > best_score_g:
                best_score_g = sc
                best_choice_g = chosen.copy()
                best_shows_g = shows.copy()
                best_meetgreet_g = meetgreet_data.copy() if meetgreet_data else None
                best_meetgreet_insert_g = meetgreet_inserted_at
                best_total_time_g = total_t
            return
        
        # Branch 1: Take project i
        chosen.append(i)
        zone = zones_arr[i]
        added = zone not in visited
        if added:
            visited.add(zone)
        
        new_pt = proj_time + times_arr[i]
        new_ps = proj_score + scores_arr[i]
        
        # Try to add a show after finishing this zone
        show_appended = False
        extra_show_t = 0
        extra_show_s = 0.0
        
        if added:
            info = pick_show(zone, new_pt)
            if info is not None:
                s_name, s_sc, s_slot, s_wait, s_dur = info
                candidate_t = new_pt + s_wait + s_dur
                st = len(visited) * SHOP_TIME
                ct = len(visited) * CHECKIN_TIME
                lunch_tmp = MEAL_TIME if not lunch_done and new_pt > MEAL_TRIGGER else 0
                dinner_tmp = DINNER_DUR if (not dinner_done and new_pt > DINNER_TRIGGER
                                            and new_pt + DINNER_DUR <= DINNER_END_LIMIT) else 0
                mg_tmp = (meetgreet_data["排队时间"] + meetgreet_data["duration"]) if (meetgreet_data and meetgreet_inserted_at) else 0
                if candidate_t + st + ct + lunch_tmp + dinner_tmp + mg_tmp <= LIMIT:
                    shows.append({
                        "园区": zone, "演出": s_name, "评分": s_sc,
                        "slot_off": s_slot, "wait": s_wait, "dur": s_dur
                    })
                    show_appended = True
                    extra_show_t = s_wait + s_dur
                    extra_show_s = s_sc
        
        # Try to insert meet-and-greet here (if not yet inserted)
        if meetgreet_data and not meetgreet_inserted_at and added:
            zone_idx = len(visited) - 1
            mg_zone = meetgreet_data["地点"]
            valid_zones = set()
            if zone_idx < len(fixed_zone_order):
                valid_zones.add(fixed_zone_order[zone_idx])
            if zone_idx + 1 < len(fixed_zone_order):
                valid_zones.add(fixed_zone_order[zone_idx + 1])
            if mg_zone in valid_zones:
                mg_queue = meetgreet_data["排队时间"]
                mg_dur = meetgreet_data["duration"]
                participate_time = new_pt + extra_show_t + mg_queue
                if meetgreet_data["开始时间"] <= participate_time <= meetgreet_data["结束时间"]:
                    mg_total = mg_queue + mg_dur
                    new_time_with_mg = new_pt + extra_show_t + mg_total
                    st = len(visited) * SHOP_TIME
                    ct = len(visited) * CHECKIN_TIME
                    lunch_tmp = MEAL_TIME if not lunch_done and new_time_with_mg > MEAL_TRIGGER else 0
                    dinner_tmp = DINNER_DUR if (not dinner_done and new_time_with_mg > DINNER_TRIGGER
                                                and new_time_with_mg + DINNER_DUR <= DINNER_END_LIMIT) else 0
                    if new_time_with_mg + st + ct + lunch_tmp + dinner_tmp <= LIMIT:
                        dfs(i + 1, chosen, shows, visited,
                            new_time_with_mg, new_ps, show_score + extra_show_s,
                            meetgreet_data, (zone_idx, new_pt + extra_show_t),
                            new_lunch_done, new_dinner_done)
        
        # Continue without inserting meet-and-greet
        dfs(i + 1, chosen, shows, visited,
            new_pt + extra_show_t, new_ps, show_score + extra_show_s,
            meetgreet_data, meetgreet_inserted_at,
            new_lunch_done, new_dinner_done)
        
        # backtrack
        if show_appended:
            shows.pop()
        if added:
            visited.remove(zone)
        chosen.pop()
        
        # Branch 2: Skip project i
        dfs(i + 1, chosen, shows, visited, proj_time, proj_score, show_score,
            meetgreet_data, meetgreet_inserted_at, lunch_done, dinner_done)
    
    # First run without meet-and-greet to get baseline
    print(f"\n--- {scenario_name} ---")
    print("Solving without meet-and-greet...")
    dfs(0, [], [], set(), 0, 0.0, 0.0, None, None, False, False)
    baseline_score = best_score_g
    print(f"Baseline score (no meet-and-greet): {baseline_score:.2f}")
    
    # Then try each meet-and-greet
    print(f"Trying {len(meetgreets)} meet-and-greets...")
    for mg in meetgreets:
        mg_with_dur = mg.copy()
        mg_with_dur["duration"] = sample_meetgreet_dur()
        temp_score = best_score_g
        dfs(0, [], [], set(), 0, 0.0, 0.0, mg_with_dur, None, False, False)
        if best_score_g > temp_score:
            print(f"  ✓ Improved with {mg['名称']} (评分 {mg['评分']:.1f}): {best_score_g:.2f}")
    
    print(f"Final best score for {scenario_name}: {best_score_g:.2f}")
    
    # Prepare return data
    result = {
        "scenario": scenario_name,
        "best_score": best_score_g,
        "best_choice": best_choice_g.copy(),
        "best_shows": best_shows_g.copy(),
        "best_meetgreet": best_meetgreet_g.copy() if best_meetgreet_g else None,
        "best_meetgreet_insert": best_meetgreet_insert_g,
        "total_time": best_total_time_g,
        "items": items,
    }
    return result


# ===========================================================================
#  Main: run three scenarios and produce comparison table
# ===========================================================================
scenarios = [
    ("项目_周中情侣.csv", "周中情侣"),
    ("项目_周末情侣.csv", "周末情侣"),
    ("项目_节假日情侣.csv", "节假日情侣")
]

results = []
for csv_file, name in scenarios:
    res = run_scenario(csv_file, name, reset_seed=True)
    results.append(res)

# Output comparison table
print("\n" + "=" * 80)
print("                           对比表")
print("=" * 80)
print(f"{'场景':<12} {'最优总满意度':<16} {'游玩项目数':<12} ")
print("-" * 80)
for res in results:
    project_count = len(res["best_choice"])
    print(f"{res['scenario']:<12} {res['best_score']:<16.2f} {project_count:<12} ")
print("=" * 80)

Loaded 12 meet-and-greets (common to all scenarios)

--- 周中情侣 ---
Solving without meet-and-greet...
Baseline score (no meet-and-greet): 129.10
Trying 12 meet-and-greets...
Final best score for 周中情侣: 129.10

--- 周末情侣 ---
Solving without meet-and-greet...
Baseline score (no meet-and-greet): 122.70
Trying 12 meet-and-greets...
Final best score for 周末情侣: 122.70

--- 节假日情侣 ---
Solving without meet-and-greet...
Baseline score (no meet-and-greet): 114.40
Trying 12 meet-and-greets...
Final best score for 节假日情侣: 114.40

                           对比表
场景           最优总满意度           游玩项目数        
--------------------------------------------------------------------------------
周中情侣         129.10           18           
周末情侣         122.70           18           
节假日情侣        114.40           15           
